# Planning and Reflection Build Guide

John Martinez

In [1]:
from copy import deepcopy

target = 3

initial = {
    "inventory": {
        "folders": 3,
        "labels": 2,
        "spares": 1
    },
    "packed": 0,
    "verified": False,
}

def goal_met(state, target):

    return (
        state["packed"] == target
        and state["verified"]
    )

print(initial)

print(
    "Goal met:",
    goal_met(
        initial,
        target
    )
)

{'inventory': {'folders': 3, 'labels': 2, 'spares': 1}, 'packed': 0, 'verified': False}
Goal met: False


## Prediction

Three kits cannot be assembled immediately because only two ready labels are available. Packed count alone is not enough because verification is also required.

In [2]:
first_plan = [
    {"action": "inspect", "quantity": 0},
    {"action": "assemble", "quantity": 3},
    {"action": "check", "quantity": 0},
]

for number, proposal in enumerate(
    first_plan,
    start=1
):
    print(number, proposal)

print(
    "State still unchanged:",
    initial["packed"] == 0
)

1 {'action': 'inspect', 'quantity': 0}
2 {'action': 'assemble', 'quantity': 3}
3 {'action': 'check', 'quantity': 0}
State still unchanged: True


## Prediction

The restock step is missing. If checking runs before successful assembly, verification should fail.

In [3]:
def inspect_stock(state, quantity, target):
    return "Stock inspected."

def restock(state, quantity, target):

    stock = state["inventory"]

    if stock["spares"] < quantity:
        raise ValueError(
            "not enough spare labels"
        )

    stock["spares"] -= quantity
    stock["labels"] += quantity

    state["verified"] = False

    return (
        f"Moved {quantity} "
        "spare label(s)."
    )

demo = deepcopy(initial)

print(
    restock(
        demo,
        1,
        target
    )
)

print(
    demo["inventory"]
)

print(
    "Original labels:",
    initial["inventory"]["labels"]
)

Moved 1 spare label(s).
{'folders': 3, 'labels': 3, 'spares': 0}
Original labels: 2


## Prediction

Using one spare label should increase ready labels to three and reduce spares to zero. The original state should remain unchanged.

In [4]:
def assemble(state, quantity, target):

    stock = state["inventory"]

    if (
        state["packed"] + quantity
        > target
    ):
        raise ValueError(
            "would exceed the target"
        )

    if (
        stock["folders"] < quantity
        or stock["labels"] < quantity
    ):
        raise ValueError(
            "not enough ready materials"
        )

    stock["folders"] -= quantity
    stock["labels"] -= quantity

    state["packed"] += quantity
    state["verified"] = False

    return (
        f"Assembled {quantity} kit(s)."
    )


def check_kits(
    state,
    quantity,
    target
):

    state["verified"] = (
        state["packed"] == target
    )

    return (
        f"Count check: "
        f"{state['verified']}"
    )

print(
    assemble(
        demo,
        3,
        target
    )
)

print(
    "Before check:",
    goal_met(
        demo,
        target
    )
)

print(
    check_kits(
        demo,
        0,
        target
    )
)

print(
    "After check:",
    goal_met(
        demo,
        target
    )
)

Assembled 3 kit(s).
Before check: False
Count check: True
After check: True


## Prediction

The kits will assemble successfully after restocking. Before the check the goal is False. After the check the goal becomes True.


In [5]:
TOOLS = {
    "inspect": inspect_stock,
    "restock": restock,
    "assemble": assemble,
    "check": check_kits,
}

In [6]:
def validate_step(step):

    if type(step) is not dict:
        raise ValueError(
            "step must be a dictionary"
        )

    if (
        set(step.keys())
        != {"action", "quantity"}
    ):
        raise ValueError(
            "unexpected keys"
        )

    action = step["action"]
    quantity = step["quantity"]

    if action not in TOOLS:
        raise ValueError(
            "unknown action"
        )

    if (
        type(quantity) is not int
        or quantity < 0
    ):
        raise ValueError(
            "quantity must be a non-negative integer"
        )

In [7]:
validate_step(
    {
        "action": "restock",
        "quantity": 1
    }
)

print("Validation passed.")

Validation passed.


In [8]:
def dispatch(step, state, target):

    validate_step(step)

    action = step["action"]
    quantity = step["quantity"]

    return TOOLS[action](
        state,
        quantity,
        target
    )

In [9]:
demo = deepcopy(initial)

print(
    dispatch(
        {
            "action": "inspect",
            "quantity": 0
        },
        demo,
        target
    )
)

print(
    dispatch(
        {
            "action": "restock",
            "quantity": 1
        },
        demo,
        target
    )
)

Stock inspected.
Moved 1 spare label(s).


## Prediction

Validation should accept known actions with valid quantities. The dispatcher should route each step to the correct tool function.

In [10]:
def execute_plan(
    plan,
    start_state,
    target
):

    state = deepcopy(
        start_state
    )

    log = []

    for number, step in enumerate(
        plan,
        start=1
    ):

        observation = dispatch(
            step,
            state,
            target
        )

        log.append(
            {
                "step": number,
                "action": step["action"],
                "observation": observation
            }
        )

    return state, log

In [12]:
try:

    state, log = execute_plan(
        first_plan,
        initial,
        target
    )

    print(
        "Goal met:",
        goal_met(
            state,
            target
        )
    )

except Exception as error:

    print(error)

not enough ready materials


In [13]:
fixed_plan = [
    {
        "action": "inspect",
        "quantity": 0
    },
    {
        "action": "restock",
        "quantity": 1
    },
    {
        "action": "assemble",
        "quantity": 3
    },
    {
        "action": "check",
        "quantity": 0
    },
]

In [14]:
state, log = execute_plan(
    fixed_plan,
    initial,
    target
)

print(
    "Packed:",
    state["packed"]
)

print(
    "Verified:",
    state["verified"]
)

print(
    "Goal met:",
    goal_met(
        state,
        target
    )
)


Packed: 3
Verified: True
Goal met: True


In [15]:
for item in log:

    print(
        item["step"],
        item["action"],
        "->",
        item["observation"]
    )

1 inspect -> Stock inspected.
2 restock -> Moved 1 spare label(s).
3 assemble -> Assembled 3 kit(s).
4 check -> Count check: True


In [16]:
def reflect(log):

    notes = []

    actions = [
        item["action"]
        for item in log
    ]

    if "restock" in actions:

        notes.append(
            "Labels were replenished before assembly."
        )

    if "check" in actions:

        notes.append(
            "Verification completed."
        )

    return notes

In [17]:
notes = reflect(log)

for note in notes:

    print(note)

Labels were replenished before assembly.
Verification completed.


## Prediction

Reflection should capture important lessons from the successful execution. Restocking enabled assembly and the final check confirmed the goal.

In [18]:
def revise_plan():

    return [
        {
            "action": "inspect",
            "quantity": 0
        },
        {
            "action": "restock",
            "quantity": 1
        },
        {
            "action": "assemble",
            "quantity": 3
        },
        {
            "action": "check",
            "quantity": 0
        },
    ]

In [19]:
new_plan = revise_plan()

for step in new_plan:

    print(step)

{'action': 'inspect', 'quantity': 0}
{'action': 'restock', 'quantity': 1}
{'action': 'assemble', 'quantity': 3}
{'action': 'check', 'quantity': 0}


In [20]:
state, log = execute_plan(
    new_plan,
    initial,
    target
)

print(
    "Goal met:",
    goal_met(
        state,
        target
    )
)

print()

for item in log:

    print(
        item["action"],
        "->",
        item["observation"]
    )

Goal met: True

inspect -> Stock inspected.
restock -> Moved 1 spare label(s).
assemble -> Assembled 3 kit(s).
check -> Count check: True


## Prediction

The revised plan should reach the goal because reflection identified the missing restock action and the revised sequence includes it before assembly.